# 18.1 Exploratory data analysis

EDA is the disciplined habit of measuring the data before asking a model to explain it.

Before optimization, descriptive statistics, correlations, and segment rates reveal tails, associations, and hidden regimes. This notebook profiles the data first, then trains one unchanged baseline across the quality ladder.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(18)
random.seed(18)


def dataquality_ladder():
    """D1..D5 over real Breast Cancer with progressively worse data quality."""
    bc = load_breast_cancer()
    X0 = bc.data.astype(float)
    y0 = bc.target
    rng = np.random.default_rng(18)
    rungs = []

    rungs.append(("D1 clean", X0.copy(), y0.copy()))

    y2 = y0.copy()
    flip = rng.random(y2.shape) < 0.15
    y2[flip] = 1 - y2[flip]
    rungs.append(("D2 15% label noise", X0.copy(), y2))

    keep_pos = np.where(y0 == 1)[0]
    keep_neg = np.where(y0 == 0)[0][:30]
    idx = np.concatenate([keep_pos, keep_neg])
    rungs.append(("D3 class imbalance", X0[idx].copy(), y0[idx].copy()))

    X4 = X0 + rng.normal(0.0, X0.std(axis=0) * 0.5, size=X0.shape)
    rungs.append(("D4 heavy feature noise", X4, y0.copy()))

    X5 = X0.copy()
    holes = rng.random(X5.shape) < 0.2
    X5[holes] = np.nan
    col_means = np.nanmean(X5, axis=0)
    X5 = np.where(np.isnan(X5), col_means, X5)
    rungs.append(("D5 20% missing (mean-imputed)", X5, y0.copy()))

    return rungs


def split_scale(X, y, seed=0):
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.35,
        random_state=seed,
        stratify=y,
    )
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    return x_train, x_test, y_train, y_test


def fit_logistic(x_train, y_train, sample_weight=None):
    model = LogisticRegression(max_iter=2000, solver="lbfgs")
    model.fit(x_train, y_train, sample_weight=sample_weight)
    return model


def preview_ladder(rungs):
    for i, (name, X, y) in enumerate(rungs, start=1):
        counts = dict(zip(*np.unique(y, return_counts=True)))
        print(f"{i}. {name}: X={X.shape}, class_counts={counts}")
    print("sample rows", np.round(rungs[0][1][:3, :4], 3).tolist())


def plot_rungs_and_metric(rungs, rows, title, ylabel="accuracy"):
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    metrics = [row["metric"] for row in rows]
    for ax, (name, X, y), metric in zip(axes[0], rungs, metrics):
        ax.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="coolwarm", alpha=0.65)
        ax.set_title(f"{name}\n{metric:.3f}")
        ax.set_xticks([])
        ax.set_yticks([])
    axes[1][0].plot(range(1, 6), metrics, marker="o")
    axes[1][0].set_title(title)
    axes[1][0].set_xlabel("data-quality rung")
    axes[1][0].set_ylabel(ylabel)
    axes[1][0].set_ylim(0.0, 1.05)
    for ax in axes[1][1:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()



## The concept, built once (D1)

EDA starts with center, spread, standardized scores, correlation, and conditional rates:

$$ar x=rac{1}{m}\sum_i x_i,\quad \sigma=\sqrt{rac{1}{m}\sum_i(x_i-ar x)^2},\quad z_i=rac{x_i-ar x}{\sigma}$$

$$r=rac{\sum_i(x_i-ar x)(y_i-ar y)}{\sqrt{\sum_i(x_i-ar x)^2\sum_i(y_i-ar y)^2}}$$

The lesson's outlier, correlation, and segment-rate numbers are asserted before we train.

In [ ]:
def profile_then_train(X, y, seed=0):
    x_train, x_test, y_train, y_test = split_scale(X, y, seed=seed)
    model = fit_logistic(x_train, y_train)
    preds = model.predict(x_test)
    first_feature = X[:, 0]
    issue_count = 0
    if abs(np.mean(first_feature) - np.median(first_feature)) > np.std(first_feature):
        issue_count += 1
    if np.max(np.abs((first_feature - np.mean(first_feature)) / np.std(first_feature))) > 2.0:
        issue_count += 1
    if abs(np.mean(y) - 0.5) > 0.25:
        issue_count += 1
    return {
        "metric": accuracy_score(y_test, preds),
        "issue_count": issue_count,
        "preds": preds,
        "y_test": y_test,
    }

Now reproduce the worked EDA values exactly to three decimals.

In [ ]:
lesson_x = np.array([2, 3, 3, 4, 5, 20], dtype=float)
lesson_mean = lesson_x.mean()
lesson_median = np.median(lesson_x)
lesson_std = lesson_x.std()
lesson_z = (20 - lesson_mean) / lesson_std
corr_x = np.array([1, 2, 3, 4, 5], dtype=float)
corr_y = np.array([2, 4, 5, 4, 5], dtype=float)
lesson_r = np.corrcoef(corr_x, corr_y)[0, 1]
slice_a = np.array([0, 0, 0, 1])
slice_b = np.array([1, 1, 1, 0])
rate_a = slice_a.mean()
rate_b = slice_b.mean()
overall_rate = np.concatenate([slice_a, slice_b]).mean()
print(round(lesson_mean, 3), round(lesson_median, 3), round(lesson_z, 3))
print(round(lesson_r, 3), round(rate_a, 3), round(rate_b, 3), round(overall_rate, 3))
assert round(lesson_mean, 3) == 6.167
assert round(lesson_median, 3) == 3.5
assert round(lesson_z, 3) == 2.211
assert round(lesson_r, 3) == 0.775
assert round(rate_a, 3) == 0.25
assert round(rate_b, 3) == 0.75
assert round(overall_rate, 3) == 0.5

## The dataset ladder

Every notebook uses the same real Breast Cancer base table and changes data quality only: clean, label-noisy, imbalanced, feature-noisy, then missing-and-imputed. The model stays fixed so movement in the curve is caused by the data.

In [ ]:
rungs = dataquality_ladder()
preview_ladder(rungs)

In [ ]:
rows = []
for name, X, y in rungs:
    result = profile_then_train(X, y)
    rows.append({"name": name, "metric": result["metric"], "issues": result["issue_count"]})
    print(f"{name:28s} accuracy={result['metric']:.3f} eda_issues={result['issue_count']}")

In [ ]:
plot_rungs_and_metric(rungs, rows, "Accuracy vs data quality")

## Pitfall on D5: averaging away segments

The global accuracy can look acceptable while a slice defined by an EDA signal fails. The fix is not a fancier model here; it is stratified reporting so the hidden slice becomes visible and action can target data collection or cleaning.

In [ ]:
name, X, y = rungs[-1]
_, x_test, _, y_test = split_scale(X, y, seed=3)
majority_label = int(np.round(y.mean()))
preds = np.full_like(y_test, majority_label)
global_acc = accuracy_score(y_test, preds)
minority_mask = y_test == 0
minority_acc = accuracy_score(y_test[minority_mask], preds[minority_mask])
majority_acc = accuracy_score(y_test[~minority_mask], preds[~minority_mask])
worst_slice = min(minority_acc, majority_acc)
print(f"wrong global-only accuracy={global_acc:.3f}")
print(f"fixed report malignant={minority_acc:.3f} benign={majority_acc:.3f} worst={worst_slice:.3f}")
assert worst_slice < global_acc

## Evaluate it + Practice

- Metric: held-out accuracy plus EDA issue count; compare with the no-skill majority-class baseline.
- Cheap sanity check: mean and median should disagree on the lesson tail example.
- Ablation: skip the slice report and the D5 worst-slice signal disappears.
- Failure signals: large mean-median gaps, z-scores above 2, or slice rates far from the global rate.

Practice prompts:
1. Change one corruption level in `dataquality_ladder()` and predict the metric direction.


In [ ]:
# Your turn: change one corruption and rerun the ladder table.


2. Add one slice check that could catch a global-average pitfall.

In [ ]:
# Your turn: define a slice and compare its metric with the global metric.


3. Replace the fixed logistic model with another CPU-safe classifier and explain whether the data-quality ordering changed.

In [ ]:
# Your turn: try a second model without changing the data-cleaning method.
